In [1]:
#test

In [2]:
import datetime
import logging  # ログ制御用
import warnings  # 警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Node.js Theme Palette)
# =====================================================================
def show_market_performances(market_dict, title="Market Performance Overview"):
    today = datetime.date.today()
    # 1年分（約370日）のデータを取得して過去騰落率を正確に計算
    start_date = today - datetime.timedelta(days=370)
    
    # Node.js テーマ風のタイトルヘッダー (Dark Background & Neon Green Text)
    title_html = f"""
    <div style='
        background-color: #181818; 
        border-left: 5px solid #68a063; 
        padding: 12px 18px; 
        border-radius: 4px; 
        margin-bottom: 25px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    '>
        <h2 style='
            margin: 0; 
            color: #539e43; 
            font-family: "Consolas", "Courier New", Monaco, monospace; 
            font-size: 22px;
            letter-spacing: 0.5px;
        '>⬢ {title}</h2>
        <span style='
            font-size: 12px; 
            color: #888888; 
            font-family: monospace;
        '>System Status: ONLINE | As of {today.strftime('%Y-%m-%d')}</span>
    </div>
    """
    display(HTML(title_html))
    
    # 数値フォーマット設定
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # Node.jsテーマに合わせたハイライト関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        # プラス（上昇）：Node.jsのネオングリーン背景
        if val >= 10:
            return 'background-color: #1e3a1e; color: #41d888; font-weight: bold; border-radius: 2px;'
        # マイナス（下落）：ダークレッド/コーラル系
        elif val <= -10:
            return 'background-color: #3b1818; color: #ff6b6b; font-weight: bold; border-radius: 2px;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # セクションヘッダー（Node Dark Gray + Green Accent）
        section_html = f"""
        <div style='
            margin-top: 20px; 
            margin-bottom: 8px; 
            color: #215732; 
            font-family: "Consolas", monospace; 
            font-weight: bold;
            font-size: 15px;
            border-bottom: 1px solid #333333;
            padding-bottom: 4px;
        '>
            <span style='color: #68a063;'>❯</span> {market_name}
        </div>
        """
        display(HTML(section_html))
        
        for name, ticker in ticker_dict.items():
            try:
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # テーブル全体のスタイル定義（Dark Node UI Style）
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .set_table_styles([
                             # テーブル全体
                             {'selector': '', 'props': [('border-collapse', 'collapse'), ('font-family', 'Consolas, monospace'), ('font-size', '13px'), ('background-color', '#121212'), ('color', '#e0e0e0')]},
                             # ヘッダー行（Dark Gray / Green Accent）
                             {'selector': 'th', 'props': [('background-color', '#1f1f1f'), ('color', '#68a063'), ('border', '1px solid #333'), ('padding', '8px 12px'), ('text-align', 'center')]},
                             # データセル
                             {'selector': 'td', 'props': [('border', '1px solid #282828'), ('padding', '7px 12px'), ('text-align', 'right')]},
                             # ホバー効果
                             {'selector': 'tr:hover', 'props': [('background-color', '#1a261a')]}
                         ])
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [3]:
MARKET_TICKERS = {
    # ==========================================
    # 1. アメリカ市場（米国株・指数・ETF）
    # ==========================================
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG",
        "Vanguard FTSE Global All-Cap UCITS ETF (VALL.L)": "VALL.L",
    },

    # ==========================================
    # 2. 日本市場（日本株・指数）
    # ==========================================
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Japan Stock (Custom)": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },

    # ==========================================
    # 3. シンガポール市場（総合・インデックス）
    # ==========================================
    "Singapore (General)": {
        "STI Index": "^STI",
        "DBS Group (金融要塞/東南アジア最大手)": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
        "SGX Group (完全独占/国内唯一の証券取引所)": "S68.SI",
        "PropNex (不動産仲介最大手/高配当)": "OYY.SI",
    },

    # ==========================================
    # 4. シンガポール：医療セクター (Healthcare)
    # ==========================================
    "Singapore Healthcare": {
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone Holdings (クリーンルーム用手袋)": "AP4.SI",
    },

    # ==========================================
    # 5. シンガポール：食品・アグリビジネス
    # ==========================================
    "Singapore Food & Agribusiness": {
        "Wilmar International (世界最大級のパーム油)": "F34.SI",
        "Olam Group (ココア・コーヒー・ナッツ等大手)": "VC2.SI",
    },

    # ==========================================
    # 6. シンガポール：コア・安定・高配当・バリュー
    # ==========================================
    "Singapore Core & Stability": {
        "Sembcorp Industries (大手公益/エネルギー)": "U96.SI",
        "Valuetronics Holdings (電子機器製造受託/無借金)": "BN2.SI",
    },
    "Singapore High Yield & Growth": {
        "Multi-Chem (ITセキュリティ/高配当)": "AWZ.SI",
        "Yangzijiang Financial Holding (金融)": "YF8.SI",
        "Yangzijiang Shipbuilding (造船/高収益)": "BS6.SI",
        "Sanli Environmental Limited": "1E3.SI",
    },
    "Singapore Recovery & Value": {
        "Micro-Mechanics (無借金/半導体消耗工具/高配当)": "508.SI",
    },

    # ==========================================
    # 7. シンガポール：建設・インフラ
    # ==========================================
    "Singapore Construction & Infrastructure": {
        "BRC Asia (鉄筋加工最大手)": "BEC.SI",
        "OKP Holdings (道路土木インフラ/無借金)": "5CF.SI",
        "ISOTeam (HDB建物修繕塗装R&R)": "5WF.SI",
    },
}



In [4]:
show_market_performances(MARKET_TICKERS, title="Market Performance Overview")

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,175.83",-2.02%,-3.25%,-5.49%,-6.36%,-3.98%,+8.03%
NASDAQ Composite,^IXIC,"23,879.20",-2.25%,-2.57%,-5.86%,-8.19%,-9.36%,+5.83%
Apple,AAPL,349.21,-0.92%,-1.01%,-1.60%,+0.36%,+13.26%,+30.48%
NVIDIA,NVDA,145.98,-3.30%,-6.51%,-12.16%,-31.21%,-32.05%,-21.55%
Microsoft,MSFT,498.30,-1.78%,-3.62%,-4.54%,-4.19%,+19.50%,+3.34%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,95.56,+0.03%,+0.43%,+0.36%,-0.23%,-0.89%,-0.58%
Vanguard FTSE Global All-Cap UCITS ETF (VALL.L),VALL.L,3.56,+0.03%,-2.22%,-3.63%,-2.56%,-28.77%,-28.77%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"59,444.21",+0.71%,-0.37%,-2.26%,-8.01%,-1.83%,+18.12%
Toyota Motor,7203.T,"3,378.00",-0.38%,-1.46%,-0.88%,+9.18%,+16.18%,+11.79%
Sony Group,6758.T,"4,365.00",+2.92%,+1.04%,+1.96%,+8.80%,+17.90%,-3.19%
Mitsubishi UFJ Financial,8306.T,"3,197.00",-1.66%,-6.38%,-8.60%,-6.82%,+4.21%,+36.03%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"5,821.00",-3.14%,-4.10%,-6.76%,-16.89%,-7.09%,+14.37%
ニデック (6594),6594.T,"1,730.00",+1.76%,+1.35%,-3.24%,-1.82%,-35.33%,-16.18%
未来工業 (7931),7931.T,"3,200.00",-1.54%,-4.33%,-5.74%,-10.61%,+7.59%,-0.07%
東部ネットワーク (9036),9036.T,"1,098.00",-1.08%,-1.79%,+0.92%,-3.43%,-16.57%,+0.66%
ニトリHD (9843),9843.T,"3,707.00",+2.06%,+0.93%,+8.20%,+12.71%,+53.38%,+42.98%
MRK HD (9980),9980.T,85.00,-1.16%,-2.30%,+1.19%,-3.41%,-14.14%,-8.71%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,079.49",+0.49%,-2.00%,-3.85%,-6.59%,+0.22%,+12.75%
DBS Group (金融要塞/東南アジア最大手),D05.SI,67.34,+0.60%,-2.83%,-4.06%,-8.75%,+10.86%,+31.86%
UOB,U11.SI,36.14,+0.03%,-3.32%,-4.32%,-13.95%,-2.08%,+11.23%
Singtel,Z74.SI,4.20,+3.96%,+5.00%,+1.20%,+0.24%,-6.39%,-10.07%
CapitaLand Ascendas REIT,A17U.SI,2.17,-0.91%,-3.56%,-1.81%,-1.81%,-12.58%,-17.93%
SGX Group (完全独占/国内唯一の証券取引所),S68.SI,20.05,+0.15%,-2.15%,-2.82%,-2.29%,-8.92%,+22.79%
PropNex (不動産仲介最大手/高配当),OYY.SI,1.58,+1.94%,-3.07%,-4.24%,-6.51%,-11.22%,-24.97%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.83,+0.60%,+0.00%,-1.76%,+1.83%,-12.57%,-11.68%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.45,+0.00%,-2.00%,+0.82%,-0.41%,-13.03%,+0.10%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,+0.00%,-4.26%,-6.25%,-18.18%,-22.41%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,3.95,+0.00%,-1.50%,-1.99%,-2.23%,+1.09%,+3.80%
First REIT (ファースト・リート),AW9U.SI,0.16,-1.83%,-6.40%,-8.00%,-8.17%,-28.13%,-34.89%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.44,-1.12%,-3.30%,-8.33%,-16.98%,-25.51%,-11.52%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.18,+0.00%,+0.00%,+0.00%,+0.31%,+19.73%,+54.65%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,+2.33%,+10.00%,+10.00%,-8.59%,-1.47%
Medtecs International (医療用消耗品),546.SI,0.09,+0.00%,-6.00%,-1.05%,-17.54%,-28.79%,-22.95%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+0.00%,-2.78%,+7.69%,+5.92%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (世界最大級のパーム油),F34.SI,2.91,-0.34%,+0.34%,-1.36%,-14.16%,-17.83%,-8.80%
Olam Group (ココア・コーヒー・ナッツ等大手),VC2.SI,0.82,+0.00%,-2.37%,-3.51%,-9.34%,-26.45%,-7.38%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Sembcorp Industries (大手公益/エネルギー),U96.SI,5.61,+0.00%,-1.23%,-2.60%,-7.12%,-7.99%,-6.93%
Valuetronics Holdings (電子機器製造受託/無借金),BN2.SI,0.89,-2.20%,-2.73%,-6.32%,-7.29%,-8.52%,+10.85%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Multi-Chem (ITセキュリティ/高配当),AWZ.SI,2.42,-0.41%,-0.41%,+0.41%,+0.83%,-31.13%,-20.86%
Yangzijiang Financial Holding (金融),YF8.SI,0.20,+0.00%,+0.00%,-2.38%,+2.50%,-16.33%,-58.59%
Yangzijiang Shipbuilding (造船/高収益),BS6.SI,4.36,-0.68%,-1.58%,-0.91%,-13.32%,+15.34%,+36.47%
Sanli Environmental Limited,1E3.SI,0.06,-1.67%,-15.71%,-23.38%,-30.59%,-66.70%,-79.33%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Micro-Mechanics (無借金/半導体消耗工具/高配当),508.SI,0.50,+0.00%,+0.00%,+0.00%,+0.00%,-33.33%,-14.50%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
BRC Asia (鉄筋加工最大手),BEC.SI,3.94,+0.00%,-2.23%,-4.37%,-4.60%,-12.88%,+0.77%
OKP Holdings (道路土木インフラ/無借金),5CF.SI,0.55,-0.90%,-5.17%,-11.29%,-12.00%,-32.35%,-0.41%
ISOTeam (HDB建物修繕塗装R&R),5WF.SI,0.05,+0.00%,-1.92%,-7.27%,-1.73%,-32.36%,-34.07%


# Gemini Prompt
以下の【分析データ】を基に、指定の【Pythonコードフォーマット】に変換して出力してください。

【出力条件】
・MARKET_TICKERS の辞書形式で出力すること。
・辞書のキーには「企業名 (ティッカー・主要特徴/指標の要約)」を含めること。
・値（Value）には「ティッカーシンボル（.SI付き）」を入れること。
・解説文や導入の挨拶は不要で、そのままコピーして使えるPythonコードのみを出力すること。

【Pythonコードフォーマット例】
MARKET_TICKERS = {
    "カテゴリ名": {
        "銘柄名 (ティッカー・特徴要約)": "ティッカー.SI",
    },
}

【分析データ】
（※ここに分析したテキストや銘柄リスト、あるいは「以下の銘柄を分析してコード化して」と入力）